# SCRESIA thesis-decision ladder — Codex definitive runner

This is the paper-facing Colab/Kaggle notebook. It implements the corrected thesis-decision story:

- **Strict Garrido-clean action contract:** `thesis_factorized = MultiDiscrete([6,3])`, one common inventory level `I_{t,S}` plus one shift level `S`.
- **Declared per-node extension:** `factorized = MultiDiscrete([6,6,6,3])` with `inventory_period_mode="per_node"`.
- **David / DKANA handoff:** `onehot_18d` is compatibility/ablation only, not the primary thesis-decision action space.
- **Main observation surface:** `env_sdm_history_reward` with `observation_version="v5"`. The agent can observe system state; it cannot control extra thesis-forbidden levers.

Run the notebook in order. Start with `RUN_PROFILE="debug"`; switch to `RUN_PROFILE="serious"` after the smoke cells finish.

In [ ]:
# 0) Experiment configuration
RUN_PROFILE = "debug"  # "debug" or "serious"

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_NAME = "scresia"

REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
OBSERVATION_VERSION = "v5"
OBSERVATION_MODE = "env_sdm_history_reward"
STOCHASTIC_PT = True
STEP_SIZE_HOURS = 168
SEED = 42

# Debug values are intentionally tiny. Serious values are paper-facing defaults.
DEBUG = {
    "max_steps": 4,
    "ladder_replications": 1,
    "l1b_screening_replications": 1,
    "l1b_top_k": 3,
    "l1b_top_replications": 1,
    "ppo_timesteps": 16,
    "eval_episodes": 1,
    "n_steps": 8,
    "batch_size": 4,
}
SERIOUS = {
    "max_steps": 260,
    "ladder_replications": 30,
    "l1b_screening_replications": 5,
    "l1b_top_k": 20,
    "l1b_top_replications": 30,
    "ppo_timesteps": 200_000,
    "eval_episodes": 20,
    "n_steps": 1024,
    "batch_size": 256,
}
CFG = DEBUG if RUN_PROFILE == "debug" else SERIOUS

RUN_L0_L1A = True
RUN_L1B_SCREENING = False  # set True after L0/L1a completes
RUN_PPO_THESIS_FACTORIZED = True
RUN_PPO_PER_NODE_EXTENSION = False  # set True only after L1b shows value

print({"RUN_PROFILE": RUN_PROFILE, **CFG})

## Claim boundary

The acceptance logic for the paper is:

1. `L0_garrido` reproduces the thesis design rows and keeps thesis horizons in the output metadata.
2. `L1a_uniform_IxS` tests the first declared relaxation: crossing a common inventory level with shifts.
3. `L1b_per_node_IxS` tests the second declared relaxation: different inventory periods at Op3, Op5, and Op9.
4. PPO must be compared against the best static policy in the same action space. It should not be described as beating Garrido unless the intermediate static baselines are reported.

In [ ]:
# 1) Portable Colab/Kaggle/local setup
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()

RUNTIME_PIP_PACKAGES = [
    "gymnasium==1.2.3",
    "stable-baselines3==2.8.0",
    "sb3-contrib==2.8.0",
    "simpy>=4.1.1",
    "pandas>=2.0",
    "numpy>=1.24",
    "scipy>=1.10",
]

def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
    base_dir = Path("/content") if IN_COLAB else Path("/kaggle/working")
    REPO_DIR = base_dir / REPO_NAME
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)])
else:
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
    REPO_DIR = Path.cwd()

sys.path.insert(0, str(REPO_DIR.parent))
sys.path.insert(0, str(REPO_DIR))
OUTPUT_ROOT = REPO_DIR / "outputs" / "benchmarks" / "codex_definitive_ladder"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR)
print("outputs:", OUTPUT_ROOT)

In [ ]:
# 2) Verify CLI support for corrected contracts
help_text = run_cmd([sys.executable, "scripts/run_thesis_decision_ppo_smoke.py", "--help"], cwd=REPO_DIR).stdout
assert "thesis_factorized" in help_text, "CLI does not expose thesis_factorized; pull latest main."
assert "inventory-period-mode" in help_text, "CLI does not expose inventory-period-mode; pull latest main."
ladder_help = run_cmd([sys.executable, "scripts/run_thesis_decision_ladder.py", "--help"], cwd=REPO_DIR).stdout
assert "L1a_uniform_IxS" in ladder_help and "L1b_per_node_IxS" in ladder_help
print("Contract checks passed.")

In [ ]:
# 3) Static ladder: L0 Garrido + L1a common I x S
ladder_label = f"{RUN_PROFILE}_l0_l1a_seed{SEED}"
ladder_dir = OUTPUT_ROOT / ladder_label
if RUN_L0_L1A:
    cmd = [
        sys.executable, "scripts/run_thesis_decision_ladder.py",
        "--label", ladder_label,
        "--output-root", str(OUTPUT_ROOT),
        "--levels", "L0_garrido", "L1a_uniform_IxS",
        "--replications", str(CFG["ladder_replications"]),
        "--seed", str(SEED),
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--observation-version", OBSERVATION_VERSION,
        "--observation-mode", OBSERVATION_MODE,
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(CFG["max_steps"]),
    ]
    if STOCHASTIC_PT:
        cmd.append("--stochastic-pt")
    run_cmd(cmd, cwd=REPO_DIR)
print("ladder_dir:", ladder_dir)

In [ ]:
# 4) Read and rank static results
import pandas as pd

summary_path = ladder_dir / "policy_summary.csv"
episodes_path = ladder_dir / "episodes.csv"
manifest_path = ladder_dir / "manifest.json"

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.sort_values(
        ["fill_rate_order_level_mean", "order_level_ret_mean", "reward_total_mean"],
        ascending=[False, False, False],
    ).head(20))
    display(summary.groupby("ladder_level")[["fill_rate_order_level_mean", "order_level_ret_mean", "reward_total_mean"]].max())
else:
    print("No summary yet; run the previous cell first.")

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps({k: manifest[k] for k in ["levels", "replications", "max_steps", "stochastic_pt"] if k in manifest}, indent=2))

In [ ]:
# 5) Optional L1b per-node screening: 6 x 6 x 6 x 3 extension
if RUN_L1B_SCREENING:
    l1b_label = f"{RUN_PROFILE}_l1b_per_node_seed{SEED}"
    cmd = [
        sys.executable, "scripts/run_thesis_decision_ladder.py",
        "--label", l1b_label,
        "--output-root", str(OUTPUT_ROOT),
        "--levels", "L1b_per_node_IxS",
        "--replications", str(CFG["ladder_replications"]),
        "--l1b-screening-replications", str(CFG["l1b_screening_replications"]),
        "--l1b-top-k", str(CFG["l1b_top_k"]),
        "--l1b-top-replications", str(CFG["l1b_top_replications"]),
        "--seed", str(SEED),
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--observation-version", OBSERVATION_VERSION,
        "--observation-mode", OBSERVATION_MODE,
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(CFG["max_steps"]),
    ]
    if STOCHASTIC_PT:
        cmd.append("--stochastic-pt")
    run_cmd(cmd, cwd=REPO_DIR)
else:
    print("Skipped L1b. Set RUN_L1B_SCREENING=True after L0/L1a is clean.")

In [ ]:
# 6) PPO adaptive E2a: strict thesis_factorized [6,3]
if RUN_PPO_THESIS_FACTORIZED:
    ppo_label = f"{RUN_PROFILE}_ppo_thesis_factorized_seed{SEED}"
    cmd = [
        sys.executable, "scripts/run_thesis_decision_ppo_smoke.py",
        "--label", ppo_label,
        "--output-root", str(OUTPUT_ROOT),
        "--action-space-mode", "thesis_factorized",
        "--inventory-period-mode", "thesis_strict",
        "--observation-mode", OBSERVATION_MODE,
        "--observation-version", OBSERVATION_VERSION,
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--train-timesteps", str(CFG["ppo_timesteps"]),
        "--eval-episodes", str(CFG["eval_episodes"]),
        "--seed", str(SEED),
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(CFG["max_steps"]),
        "--n-steps", str(CFG["n_steps"]),
        "--batch-size", str(CFG["batch_size"]),
        "--include-static-grid",
        "--eval-ai-on-garrido-cfis",
    ]
    if STOCHASTIC_PT:
        cmd.append("--stochastic-pt")
    run_cmd(cmd, cwd=REPO_DIR)
else:
    print("Skipped PPO thesis_factorized.")

In [ ]:
# 7) Optional PPO adaptive E2b: declared per-node extension [6,6,6,3]
if RUN_PPO_PER_NODE_EXTENSION:
    ppo_label = f"{RUN_PROFILE}_ppo_per_node_extension_seed{SEED}"
    cmd = [
        sys.executable, "scripts/run_thesis_decision_ppo_smoke.py",
        "--label", ppo_label,
        "--output-root", str(OUTPUT_ROOT),
        "--action-space-mode", "factorized",
        "--inventory-period-mode", "per_node",
        "--observation-mode", OBSERVATION_MODE,
        "--observation-version", OBSERVATION_VERSION,
        "--reward-mode", REWARD_MODE,
        "--risk-level", RISK_LEVEL,
        "--train-timesteps", str(CFG["ppo_timesteps"]),
        "--eval-episodes", str(CFG["eval_episodes"]),
        "--seed", str(SEED),
        "--step-size-hours", str(STEP_SIZE_HOURS),
        "--max-steps", str(CFG["max_steps"]),
        "--n-steps", str(CFG["n_steps"]),
        "--batch-size", str(CFG["batch_size"]),
        "--include-static-grid",
        "--eval-ai-on-garrido-cfis",
    ]
    if STOCHASTIC_PT:
        cmd.append("--stochastic-pt")
    run_cmd(cmd, cwd=REPO_DIR)
else:
    print("Skipped per-node PPO. This is a declared extension, not thesis strict.")

## Interpretation rule

Report the ladder in this order: L0, L1a, L1b, E2a, E2b. The paper claim should be about where value appears after explicit relaxations. DMLPA/Transformer comparisons belong after this ladder, not before it.